# Hospital Readmission Prediction Project: Initial EDA


### Author:  Ari M.


### Purpose & Goal:
### This notebook is focused on getting the cleaned diabetes dataset into a format that can actually be used for machine learning.
### Main things I’m doing in this notebook are, removing columns that probably won’t help prediction, handling categorical variables, checking class imbalance, creating a fully numeric dataset for modeling later
### The overall goal is predicting whether a patient gets readmitted within 30 days after discharge.


### Dataset used: diabetic_cleaned.csv

In [1]:
import pandas as pd
import numpy as np

In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("datascienceam/diabetic-cleaned-csv")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/datasets/datascienceam/diabetic-cleaned-csv


In [3]:
# For loading dataset on local system
# df = pd.read_csv("../data/processed/diabetic_cleaned.csv", low_memory=False)

# Loading dataset on Kaggle
import os
file_path = os.path.join(path, "diabetic_cleaned.csv")
df = pd.read_csv(file_path, low_memory=False)

df.head()

,encounter_id,patient_nbr,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,...,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted,readmitted_30
0,2278392,8222157,Caucasian,Female,[0-10),6,25,1,1,NaN,...,No,No,No,No,No,No,No,No,NO,0
1,149190,55629189,Caucasian,Female,[10-20),1,1,7,3,NaN,...,Up,No,No,No,No,No,Ch,Yes,>30,0
2,64410,86047875,AfricanAmerican,Female,[20-30),1,1,7,2,NaN,...,No,No,No,No,No,No,No,Yes,NO,0
3,500364,82442376,Caucasian,Male,[30-40),1,1,7,2,NaN,...,Up,No,No,No,No,No,Ch,Yes,NO,0
4,16680,42519267,Caucasian,Male,[40-50),1,1,7,1,NaN,...,Steady,No,No,No,No,No,Ch,Yes,NO,0


## Target variable

`readmitted_30`

- `1` = patient readmitted within 30 days
- `0` = no readmission within 30 days

At a high level, the project goal is basically trying to identify patients who are more likely to come back soon after discharge.

## Removing Identifier Columns

`encounter_id` and `patient_nbr` are basically identifiers.

I don’t want the model accidentally learning patient-specific patterns from IDs, so I’m removing them now.

Also just feels cleaner to drop them early.

In [4]:
# dropping identifier columns
df = df.drop(columns=["encounter_id", "patient_nbr"], errors="ignore")
df.head()

,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,...,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted,readmitted_30
0,Caucasian,Female,[0-10),6,25,1,1,NaN,Pediatrics-Endocrinology,41,...,No,No,No,No,No,No,No,No,NO,0
1,Caucasian,Female,[10-20),1,1,7,3,NaN,NaN,59,...,Up,No,No,No,No,No,Ch,Yes,>30,0
2,AfricanAmerican,Female,[20-30),1,1,7,2,NaN,NaN,11,...,No,No,No,No,No,No,No,Yes,NO,0
3,Caucasian,Male,[30-40),1,1,7,2,NaN,NaN,44,...,Up,No,No,No,No,No,Ch,Yes,NO,0
4,Caucasian,Male,[40-50),1,1,7,1,NaN,NaN,51,...,Steady,No,No,No,No,No,Ch,Yes,NO,0


## Removing Low-Variance Features

During EDA I noticed `examide` and `citoglipton` barely changed across rows.

Since they were almost constant, they probably won’t contribute much signal to the model.

Could technically keep them, but they seemed unnecessary.

In [5]:
df = df.drop(columns=["examide","citoglipton"], errors="ignore")
df.head()

,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,...,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted,readmitted_30
0,Caucasian,Female,[0-10),6,25,1,1,NaN,Pediatrics-Endocrinology,41,...,No,No,No,No,No,No,No,No,NO,0
1,Caucasian,Female,[10-20),1,1,7,3,NaN,NaN,59,...,Up,No,No,No,No,No,Ch,Yes,>30,0
2,AfricanAmerican,Female,[20-30),1,1,7,2,NaN,NaN,11,...,No,No,No,No,No,No,No,Yes,NO,0
3,Caucasian,Male,[30-40),1,1,7,2,NaN,NaN,44,...,Up,No,No,No,No,No,Ch,Yes,NO,0
4,Caucasian,Male,[40-50),1,1,7,1,NaN,NaN,51,...,Steady,No,No,No,No,No,Ch,Yes,NO,0


#### I already dropped them in my previous EDA notebook, so this confirms that they are not in the cleaneddataset I imported into this notebook.

## Checking the Target Distribution

Before doing encoding/modeling I wanted to check how balanced the target actually is.

In [6]:
df["readmitted_30"].value_counts(normalize=True)

readmitted_30
0    0.888401
1    0.111599
Name: proportion, dtype: float64

### Observation

Looks pretty imbalanced:

- around 89% = not readmitted
-  around 11% = readmitted within 30 days

So accuracy alone probably won’t be a very useful metric later.

I’ll probably need to use things like:

- recall
- F1 score
- maybe class weights or resampling

depending on how bad the imbalance affects performance.

## Separating the Target

I wanted to save the target separately before encoding everything else.

Mostly just to avoid accidentally leaking target info or modifying it during preprocessing.

In [7]:
# saving target separately first
target = df["readmitted_30"].copy()

In [8]:
# removing target columns from feature set
df = df.drop(columns=["readmitted", "readmitted_30"], errors="ignore")

In [9]:
print("readmitted in df:", "readmitted" in df.columns)
print("readmitted_30 in df:", "readmitted_30" in df.columns)

readmitted in df: False
readmitted_30 in df: False


## Finding Categorical Columns

Most ML models won’t work directly with object/string columns, so I need to encode them.

In [10]:
cat_cols = [col for col in df.select_dtypes(include="object").columns]
print(cat_cols)

['race', 'gender', 'age', 'payer_code', 'medical_specialty', 'diag_1', 'diag_2', 'diag_3', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed']


#### Honestly there are way more categorical columns than I expected at first glance.

#### The diagnosis columns especially look like they might get annoying later.

## Encoding Categorical Variables

For now I’m using one-hot encoding.

There are definitely more advanced ways to handle categorical variables, but I wanted to start with something interpretable and reliable before optimizing.

In [11]:
# one-hot encoding categorical columns
df_encoded = pd.get_dummies(
    df,
    columns=cat_cols,
    drop_first=True
)

df_encoded.head()

,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,...,insulin_Up,glyburide-metformin_No,glyburide-metformin_Steady,glyburide-metformin_Up,glipizide-metformin_Steady,glimepiride-pioglitazone_Steady,metformin-rosiglitazone_Steady,metformin-pioglitazone_Steady,change_No,diabetesMed_Yes
0,6,25,1,1,41,0,1,0,0,0,...,False,True,False,False,False,False,False,False,True,False
1,1,1,7,3,59,0,18,0,0,0,...,True,True,False,False,False,False,False,False,False,True
2,1,1,7,2,11,5,13,2,0,1,...,False,True,False,False,False,False,False,False,True,True
3,1,1,7,2,44,1,16,0,0,0,...,True,True,False,False,False,False,False,False,False,True
4,1,1,7,1,51,0,8,0,0,0,...,False,True,False,False,False,False,False,False,False,True


## Thoughts After Encoding

The feature count exploded pretty fast after one-hot encoding.

Not super surprising though considering:

- diagnosis codes have tons of unique values
- medical specialties also have a lot of categories
- medication columns add up quickly

Still, I’d rather start simple and then optimize later if needed instead of prematurely complicating the pipeline.

## Adding the Target Back

In [12]:
df_encoded["readmitted_30"] = target.values

print("Shape:", df_encoded.shape)

Shape: (101766, 2418)


### Observation

The dataset jumped to over 2400 columns after encoding.

Pros:

- easy to interpret
- preserves category information
- works well with many ML models

Cons:

- huge feature space
- higher memory usage
- possible overfitting risk
- training could become slower later

For now I think it’s manageable.

## Checking High-Cardinality Features

I wanted to see which columns were causing most of the feature explosion.

In [13]:
for col in ["diag_1", "diag_2", "diag_3", "medical_specialty"]:
    print(col, df[col].nunique())

diag_1 716
diag_2 748
diag_3 789
medical_specialty 72


### Observation

The diagnosis columns have several hundred unique values each.

That explains why the encoded dataset became so wide.

I considered:

- grouping diagnosis categories
- frequency encoding
- target encoding

but I decided not to go down that rabbit hole yet.

Wanted to get a baseline pipeline working first before experimenting too much.

## Missing Values Check

After encoding I wanted to make sure there weren’t any unexpected nulls left.

In [14]:
print("Missing values:", df_encoded.isnull().sum().sum())

Missing values: 0


## Final Feature Count

In [15]:
print("Final feature count:", df_encoded.shape[1])

Final feature count: 2418


#### Honestly bigger than I originally expected. But still workable on my machine.

## Memory Usage

In [16]:
# checking approximate memory usage
print(df_encoded.memory_usage(deep=True).sum() / 1e6, "MB")

254.618664 MB


### Observation

The dataframe is using around ~250 MB.

Most of that is probably coming from one-hot encoded columns.

Not terrible, but definitely something to watch once modeling starts.

Especially if I end up creating multiple training copies/splits.

In [17]:
# On local system
# df_encoded.to_csv("../data/processed/diabetic_features.csv", index=False)

# On Kaggle
df.to_csv("/kaggle/working/diabetic_features.csv", index=False)

print("Saved feature-engineered dataset")

Saved feature-engineered dataset


## Final Validation Check

In [18]:
print(df_encoded.shape)
print(df_encoded.isnull().sum().sum())
print(df_encoded["readmitted_30"].value_counts(normalize=True))

(101766, 2418)
0
readmitted_30
0    0.888401
1    0.111599
Name: proportion, dtype: float64


## Final Notes / Recap

What I ended up doing in this notebook:

- removed identifier columns
- dropped low-information features
- checked class imbalance
- separated target variable
- one-hot encoded categorical columns
- created final ML-ready dataset

Final dataset:

- ~101k rows
- ~2400 columns
- no missing values

## Things I’d Improve Later

A few areas probably worth revisiting:

1. Class imbalance

The positive class is pretty small (~11%), so model evaluation will matter a lot.

2. High-cardinality diagnosis codes

One-hot encoding worked, but it created a massive number of columns.

There’s probably a cleaner approach.

3. Feature selection

Not all 2400+ features are going to be useful.

Could eventually reduce dimensionality or remove weak predictors.

## Next Step

Next notebook will focus on model training/testing and figuring out whether this feature setup actually performs well or not.